<a href="https://colab.research.google.com/github/sirnaet/KnowledgeEngineering/blob/main/Autonomous_Trailer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autonomous Trailer: A Knowledge-Based Agent
**ICS 3202: Artificial Intelligence | Practical Exercise 4 | Group work**

This notebook turns the self-driving trailer from Practical Exercise 3 (Uniform Cost Search on the Kenya highway graph) into a **knowledge-based agent**.

| Component | What it is | Where |
|---|---|---|
| **Knowledge base (KB)** | Facts (roads, distances, fuel stops, rest stops, terrain) plus percepts (weather, closures, accidents, fuel level, hours driven) | Section 3 |
| **Inference engine** | Forward chaining over Horn-style rules, with negation-as-failure (stratified) and built-in comparisons. Every derived fact keeps a justification | Section 2 |
| **TELL / ASK interface** | The standard KB-agent cycle: TELL percepts, ASK what to do, TELL the action taken | Sections 2, 6 |
| **Planner** | Practical Exercise 3's Uniform Cost Search, now run *only over edges the KB infers are passable*, with risk penalties for hazardous edges | Section 5 |
| **Live data** | Weather from the free Open-Meteo API (no key), with an offline fallback | Section 4 |
| **User interaction** | Text chat commands (`route`, `rain`, `close`, `why`, ...) | Section 8 |

**KB-agent loop:** `percept -> TELL(KB) -> infer -> ASK(KB) -> act -> TELL(action)`

## 1. Group details
Fill in before submitting.

| # | Name | Admission No. | GitHub username |
|---|------|---------------|-----------------|
| 1 | Paulette Mbaika | 190016 | 	Paulette-max |
| 2 | Stacy Njoki Mwangi | 160077 | njokistacy |
| 3 | Brian Sanaet Memusi | 189718 | sirnaet |


In [ ]:
import heapq, itertools, json, urllib.request, urllib.parse
from collections import defaultdict

# --- Practical Exercise 3 graph (directed, distances in km) ---
kenya_highway_graph = {
    "Nairobi":  [("Naivasha", 88), ("Narok", 142)],
    "Naivasha": [("Nakuru", 68)],
    "Nakuru":   [("Kericho", 108), ("Eldoret", 158)],
    "Narok":    [("Bomet", 102)],
    "Bomet":    [("Kericho", 78), ("Kisii", 62)],
    "Kericho":  [("Kisumu", 76)],
    "Eldoret":  [("Kisumu", 117)],
    "Kisii":    [("Kisumu", 114)],
    "Kisumu":   [],
}

# Approximate town coordinates (used for the weather API)
COORDS = {
    "Nairobi": (-1.2921, 36.8219), "Naivasha": (-0.7172, 36.4310),
    "Nakuru":  (-0.3031, 36.0800), "Narok":    (-1.0783, 35.8601),
    "Bomet":   (-0.7813, 35.3416), "Kericho":  (-0.3689, 35.2863),
    "Eldoret": (0.5143, 35.2698),  "Kisii":    (-0.6817, 34.7666),
    "Kisumu":  (-0.0917, 34.7680),
}

## 2. The knowledge engine (representation + inference)
**Representation.** Knowledge is stored as *tuples*, e.g. `("road", "Nairobi", "Naivasha", 88)`. Variables start with `?`.

**Rules** are Horn-style: `IF cond1 AND cond2 ... THEN head`. A condition can be:
- a pattern, e.g. `("weather", "?b", "heavy_rain")`
- a negation `("not", pattern)` (negation as failure, so rules are run in **strata**: stratum 0 first, then stratum 1, ...)
- a built-in comparison: `("ge", "?h", 4)`, `("lt", ...)`, `("gt", ...)`

**Inference** is forward chaining to a fixed point. Base facts (told by us or by percepts) are kept separate from derived facts, so when a percept changes, we simply re-infer. Every derived fact records *which rule and which facts* produced it, which gives us explanations (`why`).

In [ ]:
BUILTINS = {
    "ge": lambda a, b: a >= b, "gt": lambda a, b: a > b,
    "le": lambda a, b: a <= b, "lt": lambda a, b: a < b,
    "eq": lambda a, b: a == b,
}

def is_var(x): return isinstance(x, str) and x.startswith("?")

def match(pattern, fact, binding):
    """Unify a pattern with a fact. Returns extended binding or None."""
    if len(pattern) != len(fact):
        return None
    b = dict(binding)
    for p, f in zip(pattern, fact):
        if is_var(p):
            if p in b and b[p] != f:
                return None
            b[p] = f
        elif p != f:
            return None
    return b

def substitute(pattern, binding):
    return tuple(binding.get(x, x) if is_var(x) else x for x in pattern)

class Rule:
    def __init__(self, name, conditions, head, stratum=0, description=""):
        self.name, self.conditions, self.head = name, conditions, head
        self.stratum, self.description = stratum, description
    def __repr__(self):
        conds = " AND ".join(str(c) for c in self.conditions)
        return f"[{self.name}] IF {conds} THEN {self.head}"

def solve(conditions, facts, binding, premises):
    """Yield (binding, premises) for every way the conditions can be satisfied."""
    if not conditions:
        yield binding, premises
        return
    c, rest = conditions[0], conditions[1:]
    if c[0] == "not":
        if not any(match(c[1], f, binding) is not None for f in facts):
            yield from solve(rest, facts, binding, premises)
    elif c[0] in BUILTINS:
        args = [binding.get(a, a) if is_var(a) else a for a in c[1:]]
        if BUILTINS[c[0]](*args):
            yield from solve(rest, facts, binding, premises)
    else:
        for f in list(facts):
            nb = match(c, f, binding)
            if nb is not None:
                yield from solve(rest, facts, nb, premises + [f])

class KnowledgeBase:
    def __init__(self):
        self.base = set()          # told facts (static knowledge + percepts)
        self.derived = set()       # inferred facts
        self.rules = []
        self.justification = {}    # derived fact -> (rule name, premises)

    # ---- TELL / RETRACT / ASK ----
    def tell(self, *fact):
        self.base.add(tuple(fact))
    def retract(self, *pattern):
        self.base = {f for f in self.base if match(tuple(pattern), f, {}) is None}
    def add_rule(self, rule):
        self.rules.append(rule)
    @property
    def facts(self):
        return self.base | self.derived
    def ask(self, *pattern):
        """Return a list of bindings that satisfy the pattern (empty list = not entailed)."""
        out = []
        for f in self.facts:
            b = match(tuple(pattern), f, {})
            if b is not None:
                out.append(b)
        return out
    def holds(self, *fact):
        return tuple(fact) in self.facts

    # ---- inference: forward chaining, stratum by stratum ----
    def infer(self):
        self.derived, self.justification = set(), {}
        for stratum in sorted({r.stratum for r in self.rules}):
            rules = [r for r in self.rules if r.stratum == stratum]
            changed = True
            while changed:
                changed = False
                for r in rules:
                    for b, prem in solve(r.conditions, self.facts, {}, []):
                        new = substitute(r.head, b)
                        if new not in self.facts:
                            self.derived.add(new)
                            self.justification[new] = (r.name, prem)
                            changed = True
        return self.derived

    def why(self, *fact, depth=0):
        fact = tuple(fact)
        pad = "  " * depth
        if fact in self.base:
            return f"{pad}{fact}  <- told/perceived\n"
        if fact in self.justification:
            rule, prem = self.justification[fact]
            s = f"{pad}{fact}  <- by {rule}\n"
            for p in prem:
                s += self.why(*p, depth=depth + 1)
            return s
        return f"{pad}{fact}  <- NOT known\n"

## 3. Building the knowledge base
### 3.1 Static facts (the world model)
We load the Practical Exercise 3 graph into the KB as `road(from, to, km)` facts and enrich it with domain knowledge the search tree alone did not have: speed limits, terrain, fuel stations and rest stops.

> **Note for the group:** the terrain, fuel-station and rest-stop facts below are *illustrative engineering assumptions* for the exercise. Verify or replace them with real data (see the API/web-scraping extension in Section 4) before treating them as ground truth.

In [ ]:
def build_static_knowledge(kb, graph=kenya_highway_graph):
    # Roads from Practical Exercise 3
    for a, nbrs in graph.items():
        kb.tell("town", a)
        for b, km in nbrs:
            kb.tell("road", a, b, km)
            kb.tell("speed_limit", a, b, 80)          # trailer/heavy vehicle limit (km/h), assumed
            kb.tell("surface", a, b, "tarmac")

    # Terrain knowledge (illustrative assumptions - verify)
    kb.tell("steep", "Nairobi", "Naivasha")     # Rift Valley escarpment section
    kb.tell("steep", "Nakuru", "Kericho")
    kb.tell("flood_prone", "Narok", "Bomet")
    kb.tell("flood_prone", "Kisii", "Kisumu")

    # Services (illustrative assumptions - verify)
    for t in ["Nairobi", "Naivasha", "Nakuru", "Narok", "Kericho", "Eldoret", "Kisumu", "Bomet"]:
        kb.tell("fuel_station", t)
    for t in ["Naivasha", "Nakuru", "Narok", "Kericho", "Eldoret", "Kisii", "Kisumu"]:
        kb.tell("rest_stop", t)
    for t in ["Naivasha", "Nakuru", "Narok", "Bomet", "Kericho", "Eldoret", "Kisii", "Kisumu"]:
        kb.tell("rest_stop", t)

    # Vehicle constants
    kb.tell("tank_range_km", 600)
    kb.tell("avg_speed_kmh", 60)
    kb.tell("max_driving_hours", 4)        # rest recommended after this many hours
    kb.tell("min_fuel_reserve_km", 150)
    return kb

### 3.2 Rules (the reasoning knowledge)
| Rule | Meaning |
|---|---|
| R1, R2 | A reported closure or accident **blocks** a road segment |
| R3 | Heavy rain at the destination end of a flood-prone road **blocks** it |
| R4 | Heavy rain on a steep segment with a **heavy load** is too dangerous: **blocked** |
| R5 | Heavy rain at the destination end (heavy load, not already blocked) makes the segment **hazardous** |
| R6 | Night driving on a steep segment is **hazardous** |
| R7 | A road that exists and is **not blocked** is **passable** (negation as failure, stratum 1) |
| R8 | At a town with a fuel station and low range: **should refuel** |
| R9 | At a town with a rest stop after too many driving hours: **should rest** |
| R10 | The remaining fuel cannot cover the next leg: **cannot reach** (forces refuel / replan) |

In [ ]:
def build_rules(kb):
    R = Rule
    # ---- Stratum 0: blocking and hazard conditions, vehicle-state conditions ----
    kb.add_rule(R("R1_closure",  [("closed", "?a", "?b")], ("blocked", "?a", "?b")))
    kb.add_rule(R("R2_accident", [("accident", "?a", "?b")], ("blocked", "?a", "?b")))
    kb.add_rule(R("R3_flood",
        [("flood_prone", "?a", "?b"), ("weather", "?b", "heavy_rain")],
        ("blocked", "?a", "?b")))
    kb.add_rule(R("R4_steep_rain_heavy",
        [("steep", "?a", "?b"), ("weather", "?b", "heavy_rain"), ("load", "heavy")],
        ("blocked", "?a", "?b")))
    kb.add_rule(R("R6_night_steep",
        [("steep", "?a", "?b"), ("time_of_day", "night")],
        ("hazardous", "?a", "?b")))
    kb.add_rule(R("R8_refuel",
        [("at", "?t"), ("fuel_station", "?t"), ("fuel_range_km", "?r"),
         ("min_fuel_reserve_km", "?m"), ("lt", "?r", "?m")],
        ("should_refuel", "?t")))
    kb.add_rule(R("R9_rest",
        [("at", "?t"), ("rest_stop", "?t"), ("hours_driven", "?h"),
         ("max_driving_hours", "?m"), ("ge", "?h", "?m")],
        ("should_rest", "?t")))
    kb.add_rule(R("R10_cannot_reach",
        [("at", "?a"), ("road", "?a", "?b", "?d"), ("fuel_range_km", "?r"), ("gt", "?d", "?r")],
        ("cannot_reach", "?a", "?b")))
    kb.add_rule(R("R9b_rest_night",
        [("at", "?t"), ("rest_stop", "?t"), ("time_of_day", "night"), ("hours_driven", "?h"),
         ("ge", "?h", 3)],
        ("should_rest", "?t")))
    kb.add_rule(R("R11_traffic_heavy",  [("traffic", "?a", "?b", "heavy")],  ("congested", "?a", "?b")))
    kb.add_rule(R("R12_traffic_severe", [("traffic", "?a", "?b", "severe")], ("blocked", "?a", "?b")))

    # ---- Stratum 1: uses negation on stratum-0 conclusions ----
    kb.add_rule(R("R5_rain_heavy",
        [("road", "?a", "?b", "?d"), ("weather", "?b", "heavy_rain"), ("load", "heavy"),
         ("not", ("blocked", "?a", "?b"))],
        ("hazardous", "?a", "?b"), stratum=1))
    kb.add_rule(R("R7_passable",
        [("road", "?a", "?b", "?d"), ("not", ("blocked", "?a", "?b"))],
        ("passable", "?a", "?b", "?d"), stratum=1))
    return kb

kb = build_rules(build_static_knowledge(KnowledgeBase()))
print(f"Static facts: {len(kb.base)} | Rules: {len(kb.rules)}")
for r in kb.rules:
    print(" ", r)

Static facts: 66 | Rules: 13
  [R1_closure] IF ('closed', '?a', '?b') THEN ('blocked', '?a', '?b')
  [R2_accident] IF ('accident', '?a', '?b') THEN ('blocked', '?a', '?b')
  [R3_flood] IF ('flood_prone', '?a', '?b') AND ('weather', '?b', 'heavy_rain') THEN ('blocked', '?a', '?b')
  [R4_steep_rain_heavy] IF ('steep', '?a', '?b') AND ('weather', '?b', 'heavy_rain') AND ('load', 'heavy') THEN ('blocked', '?a', '?b')
  [R6_night_steep] IF ('steep', '?a', '?b') AND ('time_of_day', 'night') THEN ('hazardous', '?a', '?b')
  [R8_refuel] IF ('at', '?t') AND ('fuel_station', '?t') AND ('fuel_range_km', '?r') AND ('min_fuel_reserve_km', '?m') AND ('lt', '?r', '?m') THEN ('should_refuel', '?t')
  [R9_rest] IF ('at', '?t') AND ('rest_stop', '?t') AND ('hours_driven', '?h') AND ('max_driving_hours', '?m') AND ('ge', '?h', '?m') THEN ('should_rest', '?t')
  [R10_cannot_reach] IF ('at', '?a') AND ('road', '?a', '?b', '?d') AND ('fuel_range_km', '?r') AND ('gt', '?d', '?r') THEN ('cannot_reach', '?a', 

## 4. Live data: weather via API (with offline fallback)
The agent can **perceive** the weather for each town using the free [Open-Meteo](https://open-meteo.com) API (no key needed). If there is no internet (or the API fails), the agent falls back to a scripted offline weather table so the notebook always runs.

Heavy rain is flagged when precipitation is at least 4 mm or the WMO weather code indicates heavy rain/thunderstorm (65, 67, 75, 82, 95, 96, 99).

In [ ]:
HEAVY_CODES = {65, 67, 75, 82, 95, 96, 99}
OFFLINE_WEATHER = {t: "clear" for t in COORDS}      # fallback: dry everywhere

def fetch_weather(town, timeout=6):
    """Return 'heavy_rain' or 'clear' for a town using Open-Meteo, or None on failure."""
    lat, lon = COORDS[town]
    q = urllib.parse.urlencode({"latitude": lat, "longitude": lon,
                                "current": "precipitation,weather_code"})
    try:
        with urllib.request.urlopen(f"https://api.open-meteo.com/v1/forecast?{q}", timeout=timeout) as r:
            cur = json.loads(r.read())["current"]
        heavy = cur.get("precipitation", 0) >= 4 or cur.get("weather_code") in HEAVY_CODES
        return "heavy_rain" if heavy else "clear"
    except Exception as e:
        return None

def perceive_weather(kb, use_api=True, verbose=True):
    """Percept step: refresh weather facts for every town."""
    kb.retract("weather", "?x", "?y")
    source = "API"
    for t in COORDS:
        w = fetch_weather(t) if use_api else None
        if w is None:
            w, source = OFFLINE_WEATHER[t], "offline fallback"
        kb.tell("weather", t, w)
    if verbose:
        rainy = [f[1] for f in kb.base if f[0] == "weather" and f[2] == "heavy_rain"]
        print(f"Weather source: {source}. Heavy rain at: {rainy or 'none'}")

perceive_weather(kb)

Weather source: API. Heavy rain at: none


In [ ]:
import os

def get_api_key(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

def fetch_traffic(a, b, timeout=8):
    """Return 'light' | 'moderate' | 'heavy' | 'severe' for road a->b using TomTom
    Traffic Flow Segment Data, sampled at the midpoint of the road. None on failure."""
    key = get_api_key("TrafficAPI")
    if not key:
        return None
    lat = (COORDS[a][0] + COORDS[b][0]) / 2
    lon = (COORDS[a][1] + COORDS[b][1]) / 2
    q = urllib.parse.urlencode({"key": key, "point": f"{lat},{lon}", "unit": "KMPH"})
    url = f"https://api.tomtom.com/traffic/services/4/flowSegmentData/absolute/10/json?{q}"
    try:
        with urllib.request.urlopen(url, timeout=timeout) as r:
            seg = json.loads(r.read())["flowSegmentData"]
        if seg.get("roadClosure"):
            return "severe"
        current, free = seg["currentSpeed"], seg["freeFlowSpeed"]
        ratio = free / current if current else 99
        return "light" if ratio < 1.15 else "moderate" if ratio < 1.5 else "heavy" if ratio < 2.0 else "severe"
    except Exception as e:
        print(f"   (traffic API failed for {a}->{b}: {e})")
        return None

def perceive_traffic(kb, use_api=True, verbose=True):
    """Percept step: refresh traffic(a, b, level) facts for every road."""
    kb.retract("traffic", "?a", "?b", "?l")
    source, jams = "API", []
    for a, nbrs in kenya_highway_graph.items():
        for b, _ in nbrs:
            level = fetch_traffic(a, b) if use_api else None
            if level is None:
                level, source = "light", "offline fallback (assumed light)"
            kb.tell("traffic", a, b, level)
            if level in ("heavy", "severe"):
                jams.append((a, b, level))
    if verbose:
        print(f"Traffic source: {source}. Heavy/severe: {jams or 'none'}")

## 5. The planner: Uniform Cost Search over *inferred* knowledge
This is the Practical Exercise 3 algorithm with two knowledge-driven changes:
1. The agent does **not** read the raw graph. It **ASKs the KB** for `passable(from, ?to, ?km)`, so blocked roads are never considered.
2. Edges the KB marked `hazardous` cost `km x 1.5` (a risk penalty), so safer routes can win even if slightly longer.

In [ ]:
RISK_MULTIPLIER = 1.5
CONGESTION_MULTIPLIER = 1.3

def kb_edges(kb, node):
    """ASK the KB for outgoing passable edges, with risk and traffic adjusted cost."""
    edges = []
    for b in kb.ask("passable", node, "?to", "?km"):
        mult = 1.0
        if kb.holds("hazardous", node, b["?to"]): mult *= RISK_MULTIPLIER
        if kb.holds("congested", node, b["?to"]): mult *= CONGESTION_MULTIPLIER
        edges.append((b["?to"], b["?km"], b["?km"] * mult))
    return edges

def ucs_kb(kb, start, goal):
    """UCS using only KB-inferred passable edges. Returns (cost, path, real_km) or (inf, [], 0)."""
    counter = itertools.count()
    frontier = [(0, next(counter), start)]
    best, parent, explored = {start: 0}, {start: None}, set()
    while frontier:
        g, _, node = heapq.heappop(frontier)
        if node in explored:
            continue
        if node == goal:
            path, n = [], node
            while n is not None:
                path.append(n); n = parent[n]
            path.reverse()
            km = sum(next(k for t, k, _ in kb_edges(kb, a) if t == b) for a, b in zip(path, path[1:]))
            return g, path, km
        explored.add(node)
        for nbr, km, cost in kb_edges(kb, node):
            ng = g + cost
            if nbr not in explored and ng < best.get(nbr, float("inf")):
                best[nbr], parent[nbr] = ng, node
                heapq.heappush(frontier, (ng, next(counter), nbr))
    return float("inf"), [], 0

## 6. The knowledge-based agent (TELL, infer, ASK, act)
At every town the agent:
1. **TELL**s the KB its percepts (`at`, `fuel_range_km`, `hours_driven`) and any events (closures, accidents, weather).
2. **Infers** with forward chaining.
3. **ASK**s: *should I refuel? should I rest? is my next road still passable?* If the next road is no longer passable, it **replans**.
4. **Acts** (drive, refuel, rest, replan) and **TELL**s the KB the action.

In [ ]:
class TrailerAgent:
    def __init__(self, kb, start, goal, load="heavy", fuel_range_km=600, time_of_day="day"):
        self.kb, self.pos, self.goal = kb, start, goal
        self.fuel = fuel_range_km
        self.hours = 0.0
        self.total_hours = 0.0
        self.total_km = 0
        self.log = []
        kb.retract("load", "?x"); kb.tell("load", load)
        kb.retract("time_of_day", "?x"); kb.tell("time_of_day", time_of_day)
        base = next(iter(kb.ask("avg_speed_kmh", "?v")))["?v"]
        self.avg_speed = base * (0.8 if time_of_day == "night" else 1)  # slower at night
        self.tank = next(iter(kb.ask("tank_range_km", "?v")))["?v"]

    def say(self, msg):
        self.log.append(msg); print(msg)

    def tell_percepts(self):
        kb = self.kb
        for p in ("at", "fuel_range_km", "hours_driven"):
            kb.retract(p, "?x")
        kb.tell("at", self.pos)
        kb.tell("fuel_range_km", self.fuel)
        kb.tell("hours_driven", round(self.hours, 2))
        kb.infer()

    def apply_events(self, events):
        for ev in events or []:
            self.kb.tell(*ev)
            self.say(f"   PERCEPT: {' '.join(map(str, ev))}")

    def run(self, events_at=None, max_steps=30):
        events_at = events_at or {}
        self.say(f"Mission: {self.pos} -> {self.goal} | load={next(iter(self.kb.ask('load','?x')))['?x']} | start range={self.fuel} km")
        self.apply_events(events_at.get("start"))
        for _ in range(max_steps):
            self.tell_percepts()
            if self.pos == self.goal:
                # CHANGED: reports total trip time and the current stretch since the last rest
                self.say(f"ARRIVED at {self.goal}. Distance driven: {self.total_km} km, "
                         f"total driving time: {self.total_hours:.1f} h "
                         f"(current stretch since last rest: {self.hours:.1f} h).")
                return True
            # ASK: services
            if self.kb.holds("should_refuel", self.pos):
                self.say(f"[{self.pos}] ASK should_refuel -> YES. Refuelling (range {self.fuel} -> {self.tank} km).")
                self.fuel = self.tank; self.kb.tell("did", "refuel", self.pos); self.tell_percepts()
            if self.kb.holds("should_rest", self.pos):
                self.say(f"[{self.pos}] ASK should_rest -> YES ({self.hours:.1f} h driven). Resting; hours reset.")
                self.hours = 0; self.kb.tell("did", "rest", self.pos); self.tell_percepts()
            # PLAN using KB
            cost, path, km = ucs_kb(self.kb, self.pos, self.goal)
            if not path:
                self.say(f"[{self.pos}] NO SAFE ROUTE to {self.goal}. Agent waits for conditions to change.")
                return False
            nxt = path[1]
            leg = next(k for t, k, _ in kb_edges(self.kb, self.pos) if t == nxt)
            if self.kb.holds("cannot_reach", self.pos, nxt) or leg > self.fuel:
                if self.kb.holds("fuel_station", self.pos):
                    self.say(f"[{self.pos}] Range {self.fuel} km < leg {leg} km. Refuelling before departure.")
                    self.fuel = self.tank; self.tell_percepts()
                else:
                    self.say(f"[{self.pos}] STRANDED: cannot reach {nxt} and no fuel station here."); return False
            self.say(f"[{self.pos}] Plan: {' -> '.join(path)} (risk-adjusted cost {cost:.0f}, {km} km). Next leg: {nxt} ({leg} km)")
            # ACT: drive
            self.fuel -= leg; self.total_km += leg
            self.hours += leg / self.avg_speed
            self.total_hours += leg / self.avg_speed
            self.pos = nxt
            self.kb.tell("did", "drive", nxt)
            self.apply_events(events_at.get(nxt))
        return False

## 7. Demonstrations
Each scenario builds a fresh KB so results are independent.

In [ ]:
def fresh_kb(weather_api=False):
    k = build_rules(build_static_knowledge(KnowledgeBase()))
    perceive_weather(k, use_api=weather_api, verbose=False)
    return k

print("=" * 70); print("SCENARIO 1: Baseline (should match Practical Exercise 3: 340 km)"); print("=" * 70)
TrailerAgent(fresh_kb(), "Nairobi", "Kisumu", load="light").run()

SCENARIO 1: Baseline (should match Practical Exercise 3: 340 km)
Mission: Nairobi -> Kisumu | load=light | start range=600 km
[Nairobi] Plan: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu (risk-adjusted cost 340, 340 km). Next leg: Naivasha (88 km)
[Naivasha] Plan: Naivasha -> Nakuru -> Kericho -> Kisumu (risk-adjusted cost 252, 252 km). Next leg: Nakuru (68 km)
[Nakuru] Plan: Nakuru -> Kericho -> Kisumu (risk-adjusted cost 184, 184 km). Next leg: Kericho (108 km)
[Kericho] ASK should_rest -> YES (4.4 h driven). Resting; hours reset.
[Kericho] Plan: Kericho -> Kisumu (risk-adjusted cost 76, 76 km). Next leg: Kisumu (76 km)
ARRIVED at Kisumu. Distance driven: 340 km, total driving time: 5.7 h (current stretch since last rest: 1.3 h).


True

In [ ]:
print("=" * 70); print("SCENARIO 2: Accident reported on Nakuru -> Kericho while en route"); print("=" * 70)
TrailerAgent(fresh_kb(), "Nairobi", "Kisumu", load="light").run(
    events_at={"Naivasha": [("accident", "Nakuru", "Kericho")]})

SCENARIO 2: Accident reported on Nakuru -> Kericho while en route
Mission: Nairobi -> Kisumu | load=light | start range=600 km
[Nairobi] Plan: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu (risk-adjusted cost 340, 340 km). Next leg: Naivasha (88 km)
   PERCEPT: accident Nakuru Kericho
[Naivasha] Plan: Naivasha -> Nakuru -> Eldoret -> Kisumu (risk-adjusted cost 343, 343 km). Next leg: Nakuru (68 km)
[Nakuru] Plan: Nakuru -> Eldoret -> Kisumu (risk-adjusted cost 275, 275 km). Next leg: Eldoret (158 km)
[Eldoret] ASK should_rest -> YES (5.2 h driven). Resting; hours reset.
[Eldoret] Plan: Eldoret -> Kisumu (risk-adjusted cost 117, 117 km). Next leg: Kisumu (117 km)
ARRIVED at Kisumu. Distance driven: 431 km, total driving time: 7.2 h (current stretch since last rest: 1.9 h).


True

In [ ]:
print("=" * 70); print("SCENARIO 3: Heavy rain at Kericho, HEAVY load (steep + rain = blocked)"); print("=" * 70)
k = fresh_kb(); k.retract("weather", "Kericho", "?w"); k.tell("weather", "Kericho", "heavy_rain"); k.tell("load", "heavy")
k.infer()
print("Blocked  :", sorted(f[1:] for f in k.derived if f[0] == "blocked"))
print("Hazardous:", sorted(f[1:] for f in k.derived if f[0] == "hazardous"))
print("\nWhy is Nakuru->Kericho blocked?")
print(k.why("blocked", "Nakuru", "Kericho"))
TrailerAgent(k, "Nairobi", "Kisumu", load="heavy").run()

SCENARIO 3: Heavy rain at Kericho, HEAVY load (steep + rain = blocked)
Blocked  : [('Nakuru', 'Kericho')]
Hazardous: [('Bomet', 'Kericho')]

Why is Nakuru->Kericho blocked?
('blocked', 'Nakuru', 'Kericho')  <- by R4_steep_rain_heavy
  ('steep', 'Nakuru', 'Kericho')  <- told/perceived
  ('weather', 'Kericho', 'heavy_rain')  <- told/perceived
  ('load', 'heavy')  <- told/perceived

Mission: Nairobi -> Kisumu | load=heavy | start range=600 km
[Nairobi] Plan: Nairobi -> Narok -> Bomet -> Kisii -> Kisumu (risk-adjusted cost 420, 420 km). Next leg: Narok (142 km)
[Narok] Plan: Narok -> Bomet -> Kisii -> Kisumu (risk-adjusted cost 278, 278 km). Next leg: Bomet (102 km)
[Bomet] ASK should_rest -> YES (4.1 h driven). Resting; hours reset.
[Bomet] Plan: Bomet -> Kisii -> Kisumu (risk-adjusted cost 176, 176 km). Next leg: Kisii (62 km)
[Kisii] Plan: Kisii -> Kisumu (risk-adjusted cost 114, 114 km). Next leg: Kisumu (114 km)
ARRIVED at Kisumu. Distance driven: 420 km, total driving time: 7.0 h (cu

True

In [ ]:
print("=" * 70); print("SCENARIO 4: Low fuel and long hours (refuel and rest rules fire)"); print("=" * 70)
TrailerAgent(fresh_kb(), "Nairobi", "Kisumu", load="light", fuel_range_km=200).run()

SCENARIO 4: Low fuel and long hours (refuel and rest rules fire)
Mission: Nairobi -> Kisumu | load=light | start range=200 km
[Nairobi] Plan: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu (risk-adjusted cost 340, 340 km). Next leg: Naivasha (88 km)
[Naivasha] ASK should_refuel -> YES. Refuelling (range 112 -> 600 km).
[Naivasha] Plan: Naivasha -> Nakuru -> Kericho -> Kisumu (risk-adjusted cost 252, 252 km). Next leg: Nakuru (68 km)
[Nakuru] Plan: Nakuru -> Kericho -> Kisumu (risk-adjusted cost 184, 184 km). Next leg: Kericho (108 km)
[Kericho] ASK should_rest -> YES (4.4 h driven). Resting; hours reset.
[Kericho] Plan: Kericho -> Kisumu (risk-adjusted cost 76, 76 km). Next leg: Kisumu (76 km)
ARRIVED at Kisumu. Distance driven: 340 km, total driving time: 5.7 h (current stretch since last rest: 1.3 h).


True

In [ ]:
print("=" * 70); print("SCENARIO 5: Night driving (steep sections become hazardous, risk-penalised)"); print("=" * 70)
TrailerAgent(fresh_kb(), "Nairobi", "Kisumu", load="light", time_of_day="night").run()

SCENARIO 5: Night driving (steep sections become hazardous, risk-penalised)
Mission: Nairobi -> Kisumu | load=light | start range=600 km
[Nairobi] Plan: Nairobi -> Narok -> Bomet -> Kericho -> Kisumu (risk-adjusted cost 398, 398 km). Next leg: Narok (142 km)
[Narok] Plan: Narok -> Bomet -> Kericho -> Kisumu (risk-adjusted cost 256, 256 km). Next leg: Bomet (102 km)
[Bomet] ASK should_rest -> YES (5.1 h driven). Resting; hours reset.
[Bomet] Plan: Bomet -> Kericho -> Kisumu (risk-adjusted cost 154, 154 km). Next leg: Kericho (78 km)
[Kericho] Plan: Kericho -> Kisumu (risk-adjusted cost 76, 76 km). Next leg: Kisumu (76 km)
ARRIVED at Kisumu. Distance driven: 398 km, total driving time: 8.3 h (current stretch since last rest: 3.2 h).


True

## 8. User interaction (chat interface)
A simple command interface lets the user **TELL** the agent new facts and **ASK** it questions.

| Command | Effect |
|---|---|
| `route A to B` | Plan and drive the trailer from A to B |
| `rain TOWN` / `clear TOWN` | TELL heavy-rain / clear weather at a town |
| `close A B` / `open A B` | TELL a road closure / reopen it |
| `accident A B` | TELL an accident on a road |
| `load heavy` / `load light` | Set load type |
| `night` / `day` | Set time of day |
| `live weather` | Fetch weather from the API |
| `live traffic` | Fetches traffic from API |
| `status` | Show blocked and hazardous roads |
| `why A B` | Explain why a road is blocked/hazardous/passable |
| `help`, `quit` | |

In [ ]:
def chat():
    import re
    k = fresh_kb(); state = {"load": "heavy", "tod": "day"}
    towns = {t.lower(): t for t in COORDS}
    HELP = ("route A to B [night|day] [heavy|light] | rain T | clear T | close A B | open A B | "
            "accident A B | load heavy/light | time night/day | live weather | status | why A B | quit")
    print("Autonomous Trailer KB agent. Type 'help' for commands, 'quit' to exit.")
    while True:
        try: cmd = input("you> ").strip()
        except EOFError: break
        words = re.findall(r"[a-z]+", cmd.lower())        # ignores commas, punctuation, case
        if not words: continue
        c = words[0]
        ts = [towns[w] for w in words[1:] if w in towns]  # towns mentioned, in order
        unknown = [w for w in words[1:] if w not in towns and w not in
                   {"to","too","from","time","load","at","in","on","night","day","heavy","light","weather","and","the"}]
        # inline modifiers, e.g. "route narok to kisumu, time night, heavy"
        if "night" in words[1:]: state["tod"] = "night"
        if "day" in words[1:]: state["tod"] = "day"
        if "heavy" in words[1:]: state["load"] = "heavy"
        if "light" in words[1:]: state["load"] = "light"
        def need(n):
            if len(ts) < n:
                print(f"I need {n} known town name(s). Towns: {', '.join(COORDS)}" + (f" (did not understand: {unknown})" if unknown else ""))
                return False
            return True
        def sync():
            k.retract("load", "?x"); k.tell("load", state["load"])
            k.retract("time_of_day", "?x"); k.tell("time_of_day", state["tod"]); k.infer()
        try:
            if c == "quit": break
            elif c == "help": print(HELP)
            elif c == "route":
                if need(2):
                    sync()
                    TrailerAgent(k, ts[0], ts[1], load=state["load"], time_of_day=state["tod"]).run()
            elif c in ("rain", "clear"):
                if need(1):
                    k.retract("weather", ts[0], "?w"); k.tell("weather", ts[0], "heavy_rain" if c == "rain" else "clear"); print("TELL ok")
            elif c in ("close", "accident"):
                if need(2):
                    if not k.ask("road", ts[0], ts[1], "?d"):
                        print(f"There is no direct road {ts[0]} -> {ts[1]} in the KB. Roads are directed, e.g. 'accident nakuru kericho'.")
                    else:
                        k.tell("closed" if c == "close" else "accident", ts[0], ts[1]); print("TELL ok")
            elif c == "open":
                if need(2): k.retract("closed", ts[0], ts[1]); k.retract("accident", ts[0], ts[1]); print("TELL ok")
            elif c in ("load", "time", "night", "day", "heavy", "light"):
                print(f"load = {state['load']}, time = {state['tod']}")
            elif c == "live":
                if "traffic" in words[1:]: perceive_traffic(k)
                else: perceive_weather(k)
            elif c == "status":
                sync()
                print(f"load = {state['load']}, time = {state['tod']}")
                print("Blocked  :", sorted(f[1:] for f in k.derived if f[0] == "blocked") or "none")
                print("Hazardous:", sorted(f[1:] for f in k.derived if f[0] == "hazardous") or "none")
            elif c == "why":
                if need(2):
                    sync(); a, b = ts[0], ts[1]
                    for pred in ("blocked", "hazardous"):
                        if k.holds(pred, a, b): print(k.why(pred, a, b)); break
                    else:
                        d = next(iter(k.ask("road", a, b, "?d")), None)
                        print(k.why("passable", a, b, d["?d"]) if d else f"No road from {a} to {b} in the KB.")
            elif c == "traffic": perceive_traffic(k)
            elif c == "jam":
                if need(2): k.tell("traffic", ts[0], ts[1], "severe"); print("TELL ok")
            else: print("Unknown command. Type 'help'.")
        except Exception as e:
            print("Could not process that command:", e)

chat()

Autonomous Trailer KB agent. Type 'help' for commands, 'quit' to exit.
you> accident nairobi naivasha
TELL ok
you> live traffic
Traffic source: API. Heavy/severe: [('Nairobi', 'Naivasha', 'severe')]
you> quit


## 9. Reflection: how this is a knowledge-based agent
- **Knowledge is separate from the program.** Roads, services and safety rules live in the KB as data. To change behaviour you edit facts/rules, not the planner.
- **Inference derives new knowledge.** `blocked`, `hazardous`, `passable`, `should_refuel`, `should_rest` are never typed in; they are inferred from facts and percepts.
- **The agent can explain itself.** `kb.why(...)` prints the rule and premises behind any conclusion.
- **Search + logic combine.** UCS from Practical Exercise 3 finds the optimal route, but only over what the KB says is safe.
- **Limitations and future work:** terrain/fuel/rest facts are assumptions to be verified; the graph is directed as in Practical Exercise 3 (add reverse edges for return trips); weather is only sampled at towns, not along the road.
- **NOTE:** For TrafficAPI, go to [TomTom API](https://my.tomtom.com/), create an account, create new key, choose traffic API, copy the key, add it to secrets in the notebook, allow notebook access.

